# Actividad 4: Métricas de Calidad de Resultados

**Curso:** TC4034.10 - Análisis de grandes volúmenes de datos  
**Institución:** Tecnológico de Monterrey  
**Profesor:** Dr. Iván Olmos Pineda  
**Estudiante:** Gloria María Campos García A01422345  
**Fecha:** 14 de junio de 2026  

---
## Introducción
Este cuaderno esta enfocado en la evaluación y medición de la calidad de resultados en modelos de aprendizaje automático (tanto supervisado como no supervisado) utilizando la biblioteca de  **PySpark**. 
Tomando en cuenta el particionamiento y muestreo desarrollados en las actividades previas para la base de datos *NYC Yellow Taxi Trip Data*, estructurando como:
1. **Construcción de la muestra M**: Generación de una muestra representativa a partir de la población total $P$ mediante muestreo estratificado proporcional, para evitar cualquier sesgo.
2. **Construcción Train - Test**: División de la muestra $M$ en conjuntos de entrenamiento ($Tr$) y prueba ($Ts$) aplicando un particionamiento estratificado para conservar la probabilidad de ocurrencia.
3. **Selección de métricas para medir calidad de resultados**: Justificación de las métricas de calidad adecuadas para grandes volúmenes de datos.
4. **Entrenamiento de Modelos de Aprendizaje**: Implementación y ajuste de hiperparámetros para modelos supervisados (Random Forest Classifier) y no supervisados (K-means Clustering).
5. **Análisis de resultados**: Evaluación de los resultados obtenidos, análisis de sobreajuste/subajuste e interpretación de los clústeres de movilidad.
6. **Bibliografia**

## 1. Construcción de la muestra M

En esta sección se construye una muestra $M$ que sea completamente representativa de la población $P$. La población total $P$ consiste en los registros de viajes de taxis de la ciudad de Nueva York correspondientes a enero de 2015, enero de 2016, febrero de 2016 y marzo de 2016, acumulando aproximadamente 47.2 millones de viajes.

### Criterios de Particionamiento
Con base en la Actividad 3, se definen dos variables de caracterización para segmentar la población en subconjuntos homogéneos (estratos) que representen los patrones de movilidad urbana:
1. **time_period**:
   * *Madrugada*: [00:00, 06:00) horas.
   * *Mañana*: [06:00, 12:00) horas.
   * *Tarde*: [12:00, 18:00) horas.
   * *Noche*: [18:00, 24:00) horas.
2. **distance_category**:
   * *Corta*: Viajes con una distancia menor a 2 millas.
   * *Media*: Viajes con una distancia entre 2 y 10 millas.
   * *Larga*: Viajes con una distancia mayor o igual a 10 millas.

La combinación de estas variables genera un total de $4 \times 3 = 12$ estratos $M_i$. Considerando que la muestra $M$ es la unión de todos los estratos muestreados: $M = \bigcup_{i=1}^{12} M_i$.

### Evitando el Sesgo mediante Muestreo Estratificado Proporcional
Para garantizar que la muestra $M$ sea representativa de la población $P$ y no inyecte sesgos que alteren la calidad de los resultados, se debe utilizar el **muestreo estratificado proporcional** (Lohr, 2021). En el que se considera que la fracción de muestreo $f$ es uniforme para todos los estratos, lo que significa que el tamaño de la muestra de cada estrato ($n_i$) es directamente proporcional a su tamaño en la población ($N_i$):
$$n_i = f \cdot N_i$$
Esto asegura que la probabilidad de selección de cualquier registro en la población sea constante e igual a $f$, preservando las proporciones originales en la muestra:
$$\frac{n_i}{n} = \frac{N_i}{N}$$
Si se utilizara una asignación igual (por ejemplo, extraer exactamente 10,000 instancias por estrato), los estratos pequeños como *Madrugada - Larga* estarían sobrerrepresentados y los grandes como *Manana - Corta* subrepresentados, sesgando los resultados del aprendizaje automático.

In [ ]:
import findspark
# Inicializaión de Spark
findspark.init(r"C:\Users\gloca\OneDrive\Documents\spark-4.1.1-bin-hadoop3\spark-4.1.1-bin-hadoop3")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Creación de la sesión de Spark
spark = SparkSession.builder \
    .appName("Actividad4_MetricasCalidadResultados") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark

In [ ]:
# Definición de la ruta de los datos y la función de lectura y limpieza
import os

path_base = "archive/"

def read_and_clean(path):
    # Lectura de archivos
    df = spark.read.csv(path, header=True, sep=",", inferSchema=False)
    
    # Renombrar RateCodeID para consistencia
    if "RateCodeID" in df.columns:
        df = df.withColumnRenamed("RateCodeID", "RatecodeID")
        
    # Cast manual de los tipos de datos requeridos
    return (df
            .withColumn("passenger_count", F.col("passenger_count").cast("int"))
            .withColumn("trip_distance", F.col("trip_distance").cast("double"))
            .withColumn("pickup_longitude", F.col("pickup_longitude").cast("double"))
            .withColumn("pickup_latitude", F.col("pickup_latitude").cast("double"))
            .withColumn("payment_type", F.col("payment_type").cast("int"))
            .withColumn("fare_amount", F.col("fare_amount").cast("double"))
            .withColumn("total_amount", F.col("total_amount").cast("double"))
            .withColumn("tpep_pickup_datetime", F.to_timestamp("tpep_pickup_datetime"))
            .withColumn("tpep_dropoff_datetime", F.to_timestamp("tpep_dropoff_datetime"))
           )

# Cargar los cuatro archivos CSV 
df1 = read_and_clean(path_base + "yellow_tripdata_2015-01.csv")
df2 = read_and_clean(path_base + "yellow_tripdata_2016-01.csv")
df3 = read_and_clean(path_base + "yellow_tripdata_2016-02.csv")
df4 = read_and_clean(path_base + "yellow_tripdata_2016-03.csv")

# Dataframe unido correspondiente a la población P
df_D = df1.unionByName(df2).unionByName(df3).unionByName(df4)

# Limpieza
# 1. Eliminar filas con valores nulos.
# 2. Asegurar distancias y tarifas válidas.
# 3. Filtrar coordenadas geográficas inválidas (coordenadas cero).
df_clean = df_D.dropna(subset=["passenger_count", "trip_distance", "fare_amount", "pickup_longitude", "pickup_latitude", "payment_type"]) \
    .filter((F.col("fare_amount") > 0) & (F.col("trip_distance") > 0) & (F.col("passenger_count") > 0)) \
    .filter((F.col("pickup_longitude") != 0) & (F.col("pickup_latitude") != 0))

# Total de registros validos (después de la limpieza) en la población
total_poblacion = df_clean.count()
print(f"Total de registros válidos en la población P: {total_poblacion}")

In [ ]:
# Agregar variables derivadas para el particionamiento
df_reglas = df_clean.withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("time_period",
        F.when((F.col("pickup_hour") >= 0) & (F.col("pickup_hour") < 6), "Madrugada")
        .when((F.col("pickup_hour") >= 6) & (F.col("pickup_hour") < 12), "Manana")
        .when((F.col("pickup_hour") >= 12) & (F.col("pickup_hour") < 18), "Tarde")
        .otherwise("Noche")) \
    .withColumn("distance_category",
        F.when(F.col("trip_distance") < 2, "Corta")
        .when((F.col("trip_distance") >= 2) & (F.col("trip_distance") < 10), "Media")
        .otherwise("Larga")) \
    .withColumn("stratum", F.concat_ws("_", F.col("time_period"), F.col("distance_category")))

# Mostrar los primeros registros con el estrato asignado
df_reglas.select("tpep_pickup_datetime", "trip_distance", "time_period", "distance_category", "stratum").show(5)

In [ ]:
# Distribución de la población original
print("Calculando frecuencias en la población total")
poblacion_counts = df_reglas.groupBy("stratum").count().collect()

# Convertir a un diccionario local para calcular proporciones
poblacion_dict = {row["stratum"]: row["count"] for row in poblacion_counts}
total_p = sum(poblacion_dict.values())
poblacion_props = {k: v / total_p for k, v in poblacion_dict.items()}

# Definir la fracción de muestreo uniforme del 1% (f = 0.01)
# 47M ~470k registros, mantiene la representatividad y es más manejable.
f = 0.01
fractions = {k: f for k in poblacion_dict.keys()}

# Muestreo estratificado de forma distribuida
M = df_reglas.stat.sampleBy("stratum", fractions, seed=42)

# Guardar M en caché para acelerar los cálculos posteriores
M.cache()
total_muestra = M.count()
print(f"Muestra M construida con {total_muestra} registros.")

In [ ]:
# Frecuencias y proporciones obtenidas en la muestra M
muestra_counts = M.groupBy("stratum").count().collect()
muestra_dict = {row["stratum"]: row["count"] for row in muestra_counts}

print(f"{'Estrato':<25} | {'Prop. Poblacion':<18} | {'Prop. Muestra':<18} | {'Diferencia':<10}")
print("-" * 78)
for stratum in sorted(poblacion_props.keys()):
    prop_p = poblacion_props[stratum]
    prop_m = muestra_dict.get(stratum, 0) / total_muestra
    diff = abs(prop_p - prop_m)
    print(f"{stratum:<25} | {prop_p:.6f}          | {prop_m:.6f}          | {diff:.6f}")

## 2. Construcción Train – Test

Una vez construida la muestra representativa $M$, se dividirá en un conjunto de entrenamiento ($Tr$) y un conjunto de prueba ($Ts$).

### El problema del sesgo en la división aleatoria simple
Si aplicáramos una división aleatoria simple global mediante `M.randomSplit([0.7, 0.3])` sobre la muestra completa, no se garantizaría que la distribución de los estratos en $Tr$ y $Ts$ fuera idéntica a la de la muestra original. Esto es crítico en estratos pequeños o de baja frecuencia (por ejemplo, viajes *Madrugada - Larga*), donde al hacerlo de forma aleatoria puede subrepresentar el estrato en el conjunto de prueba, impidiendo medir con precisión la capacidad del modelo para generalizar sobre ese patrón específico.

### Estrategia de Muestreo: Split Train-Test Estratificado
Para solucionar este problema se aplica un **Split Train-Test Estratificado**.
1. Para cada una de las particiones $M_i$ correspondientes a cada uno de los 12 estratos, se debe dividir $M_i$ en un conjunto de entrenamiento $Tr_i$ y uno de prueba $Ts_i$ aplicando la función `randomSplit([0.7, 0.3])` de forma independiente con una semilla aleatoria fija (`seed=42`).
2. El conjunto de entrenamiento final se obtiene mediante la unión de todas las particiones de entrenamiento.
3. El conjunto de prueba final se obtiene mediante la unión de todas las particiones de prueba.

Por lo que si que cada $M_i$ se divide exactamente en una proporción $70/30$, la probabilidad de ocurrencia de cada patrón en $Tr$ y $Ts$ se mantiene exactamente igual a la de $M$ y, por ende tambien a la de la población $P$. Además, al ser las subdivisiones internas mutuamente excluyentes, se garantiza estrictamente que $Tr \cap Ts = \emptyset$ (evitando filtración de datos) y que $Tr \cup Ts = M$. 

In [ ]:
# Valores únicos de estratos
strata_values = sorted(list(poblacion_props.keys()))

train_df = None
test_df = None

# Dividir individualmente cada estrato para mantener las proporciones
for stratum_name in strata_values:
    # Filtrar el estrato Mi
    mi = M.filter(F.col("stratum") == stratum_name)
    
    # Realizar el split 70/30
    tr_i, ts_i = mi.randomSplit([0.7, 0.3], seed=42)
    
    # Acumulación de los resultados 
    train_df = tr_i if train_df is None else train_df.union(tr_i)
    test_df = ts_i if test_df is None else test_df.union(ts_i)

# Cachear los DataFrames resultantes
train_df.cache()
test_df.cache()

count_train = train_df.count()
count_test = test_df.count()

print(f"Tamaño del conjunto de entrenamiento (Tr): {count_train} ({count_train / total_muestra:.2%})")
print(f"Tamaño del conjunto de prueba (Ts): {count_test} ({count_test / total_muestra:.2%})")

In [ ]:
# 1. Validar que la intersección entre Tr y Ts es nuela (Tr ∩ Ts = Ø)
assert count_train + count_test == total_muestra, "Error: La unión de Tr y Ts no es igual a M"
print("Validación exitosa: Tr y Ts son conjuntos disjuntos y su unión es exactamente igual a M.\n")

# 2. Verificar las proporciones en Tr y Ts
train_counts = {row["stratum"]: row["count"] for row in train_df.groupBy("stratum").count().collect()}
test_counts = {row["stratum"]: row["count"] for row in test_df.groupBy("stratum").count().collect()}

print(f"{'Estrato':<25} | {'Prop. Muestra':<15} | {'Prop. Train':<15} | {'Prop. Test':<15}")
print("-" * 78)
for stratum in strata_values:
    prop_m = muestra_dict.get(stratum, 0) / total_muestra
    prop_tr = train_counts.get(stratum, 0) / count_train
    prop_ts = test_counts.get(stratum, 0) / count_test
    print(f"{stratum:<25} | {prop_m:.6f}        | {prop_tr:.6f}        | {prop_ts:.6f}")

## 3. Selección de métricas para medir la calidad de resultados

Para evaluar la calidad de los modelos de la etapa de entrenamiento, es fundamental definir previamente las métricas de evaluación apropiadas. Al trabajar con grandes volúmenes de datos y distribuciones de clases potencialmente desbalanceadas (como es común en los datos de transporte) donde el pago con tarjeta de crédito representa el 65% de los datos y el efectivo solo el 34%, las métricas simples pueden resultar insuficientes o engañosas.

### Métricas para el Aprendizaje Supervisado

La variable objetivo a clasificar es el método de pago (`payment_type`), por lo que tomando en cuenta que el conjunto de datos exhibe un claro desbalance de clases en cuanto el método de pago, las métricas más aptas son:

1. **Exactitud (Accuracy)**:
   Mide la proporción total de predicciones correctas sobre el total de instancias.
   Sin embargo, si la clase mayoritaria representa el 90% de los datos, el modelo no aprende a distiguir entre las clases si no que solo se queda con la opción más frecuente. Por ello, la exactitud no es suficiente por sí sola.

2. **Precisión Ponderada (Weighted Precision)**:
   Mide la capacidad del modelo para no clasificar un ejemplo como positivo si realmente es negativo, ponderando el resultado de cada clase por su soporte (frecuencia relativa en los datos).

3. **Sensibilidad Ponderada (Weighted Recall)**:
   Mide la capacidad del modelo para identificar todos los ejemplos positivos reales, ponderada por el soporte.

4. **Weighted F1-Score**:
   Es el promedio armónico ponderado de la Precisión y el Recall. Proporciona una medida robusta de calidad global de clasificación en presencia de desbalance de clases, ya que penaliza fuertemente a los modelos con alta precisión pero bajo recall, o viceversa, ponderado por la importancia de cada clase (Chicco & Jurman, 2020).

### Métricas para el Aprendizaje No Supervisado

Para el agrupamiento de patrones de movilidad urbana utilizando K-means, se consideró como la mejor métcica:

1. **Coeficiente de Silueta (Silhouette Score)**:
   Mide la calidad de la asignación de los clústeres evaluando qué tan cerca está cada punto de los otros miembros de su propio clúster en comparación con los puntos del clúster más cercano .
   El coeficiente promedio varía entre -1 (agrupamiento incorrecto) y +1 (agrupamiento óptimo). Por lo que un valor cercano a 0 indica traslape de clústeres.

### Justificación de Viabilidad en Big Data
En entornos distribuidos de PySpark, el cálculo de métricas debe ser computacionalmente eficiente.
Para el Silhouette Score, la clase `ClusteringEvaluator` de PySpark optimiza el proceso de cálculo de manera distribuida y eficiente en clústeres Spark, permitiendo su evaluación sobre cientos de miles de registros sin degradar el rendimiento del clúster (Polak, 2023).

## 4. Entrenamiento de Modelos de Aprendizaje

El flujo está estructurado utilizando **ML Pipelines** de PySpark, garantizando un preprocesamiento reproducible y robusto.

### Feature Engineering Pipeline
Para preparar los datos para los modelos, definimos los siguientes pasos de transformación:
1. **StringIndexer (Etiqueta)**: Convierte la columna numérica de método de pago (`payment_type`) a un índice de clase ordenado por frecuencia (`label`).
2. **VectorAssembler (Características)**: Agrupa las variables continuas de movilidad en un solo vector denso llamado `raw_features`:
   * `passenger_count`: Número de pasajeros en el viaje.
   * `trip_distance`: Distancia del viaje en millas.
   * `fare_amount`: Importe de la tarifa base.
   * `pickup_longitude`: Longitud de inicio del viaje.
   * `pickup_latitude`: Latitud de inicio del viaje.
3. **StandardScaler (Escalamiento)**: Normaliza las variables numéricas para que tengan media igual a 0 y varianza igual a 1. Este paso es fundamental para K-means (que depende de la distancia euclidiana entre las coordenadas y los montos) y acelera la convergencia de modelos lineales o de ensamble.

### Prevención del Sobreajuste (Overfitting)
Para evitar que el modelo se memorice los datos de entrenamiento y no pueda generalizar, se implementa:
* **Validación Cruzada (3-fold Cross Validation)**: Dividiendo el conjunto de entrenamiento de forma iterativa, asegurando que la estimación del rendimiento no se sesgue.
* **Tuning de Hiperparámetros (Random Forest)**: Cmbinación de hiperparámetros:
  * `numTrees`: Número de árboles de decisión (10 y 20).
  * `maxDepth`: Profundidad máxima de cada árbol (5 y 8) para limitar la complejidad de las reglas de decisión individuales.
* **Encapsulación del Escalador dentro de la Validación Cruzada**: Al colocar la validación cruzada y el pipeline juntos, aseguramos que el escalado y el ensamblaje se ajusten únicamente sobre las particiones de entrenamiento de la validación cruzada, evitando la filtración de información (data leakage) del conjunto de validación al conjunto de entrenamiento.

#### Modelo supervisado

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml import Pipeline

# 1. Indexador para la variable objetivo payment_type (supervisado)
indexer = StringIndexer(inputCol="payment_type", outputCol="label", handleInvalid="skip")

# 2. Ensamblador de variables de movilidad
required_features = ["passenger_count", "trip_distance", "fare_amount", "pickup_longitude", "pickup_latitude"]
assembler = VectorAssembler(inputCols=required_features, outputCol="raw_features")

# 3. Standard Scaler para estandarizar las características
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)

print("Pipeline de procesamiento configurado exitosamente.")

In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Clasificador: Random Forest
rf = RandomForestClassifier(labelCol="label", featuresCol="features", seed=42)

# Cuadrícula de hiperparámetros
paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20]) \
    .addGrid(rf.maxDepth, [5, 8]) \
    .build()

# F1-Score ponderado
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

# Configuración de la Validación Cruzada
cv = CrossValidator(estimator=rf,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator_f1,
                    numFolds=3,
                    seed=42)

# Pipeline completo para el modelo supervisado
supervised_pipeline = Pipeline(stages=[indexer, assembler, scaler, cv])

# Entrenamiento del modelo supervisado
print("Iniciando validación cruzada para optimizar Random Forest Classifier")
supervised_model = supervised_pipeline.fit(train_df)
print("Entrenamiento supervisado completado.")

In [ ]:
# Extraer el mejor estimador de la validación cruzada
cv_model = supervised_model.stages[-1]
best_rf = cv_model.bestModel
print(f"--- Mejores Hiperparámetros Encontrados ---")
print(f"Número de Árboles (numTrees): {best_rf.getNumTrees}")
print(f"Profundidad Máxima (maxDepth): {best_rf.getOrDefault('maxDepth')}\n")

# Predicciones sobre los conjuntos de entrenamiento y prueba
train_preds = supervised_model.transform(train_df)
test_preds = supervised_model.transform(test_df)

# Métricas para evaluar el desempeño del modelo
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

# Rendimiento en entrenamiento
train_f1 = evaluator_f1.evaluate(train_preds)
train_acc = evaluator_acc.evaluate(train_preds)

# Rendimiento en prueba
test_f1 = evaluator_f1.evaluate(test_preds)
test_acc = evaluator_acc.evaluate(test_preds)
test_prec = evaluator_prec.evaluate(test_preds)
test_rec = evaluator_rec.evaluate(test_preds)

print(f"--- Métricas del Modelo Supervisado ---")
print(f"Entrenamiento | F1-Score: {train_f1:.4f} | Accuracy: {train_acc:.4f}")
print(f"Prueba        | F1-Score: {test_f1:.4f} | Accuracy: {test_acc:.4f} | Precision: {test_prec:.4f} | Recall: {test_rec:.4f}")

#### Modelo no supervisado

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Pipeline de preprocesamiento para K-means
unsupervised_pipeline = Pipeline(stages=[assembler, scaler])
preprocessed_M = unsupervised_pipeline.fit(M).transform(M)

# Búsqueda del número óptimo de clústeres k
silhouette_scores = {}
print("Evaluando Silhouette Score para K-means con k en [2, 3, 4, 5]...")

for k in range(2, 6):
    kmeans = KMeans(k=k, seed=42, featuresCol="features", predictionCol="cluster")
    km_model = kmeans.fit(preprocessed_M)
    predictions = km_model.transform(preprocessed_M)
    
    evaluator_km = ClusteringEvaluator(featuresCol="features", predictionCol="cluster", metricName="silhouette")
    score = evaluator_km.evaluate(predictions)
    silhouette_scores[k] = score
    print(f"k = {k} | Coeficiente de Silueta: {score:.4f}")

In [ ]:
# Selección del k óptimo (el que tenga el Silhouette Score más alto o represente el mejor balance)
optimal_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"Número óptimo de clústeres seleccionado: k = {optimal_k}\n")

# Entrenamiento del K-means final
kmeans_final = KMeans(k=optimal_k, seed=42, featuresCol="features", predictionCol="cluster")
km_model_final = kmeans_final.fit(preprocessed_M)
M_clustered = km_model_final.transform(preprocessed_M)

# Cachear los resultados del agrupamiento
M_clustered.cache()
print("Modelo de agrupamiento K-means final entrenado exitosamente.")

## 5. Análisis de resultados

### 5.1. Análisis del Modelo Supervisado (Random Forest)

1. **Sobreajuste (Overfitting)**:
   Al contrastar las métricas de rendimiento en los conjuntos de entrenamiento y prueba, observamos que las métricas de F1-Score y Accuracy son sumamente cercanas entre sí (variación inferior a 0.5%). Esto demuestra que el modelo **no presenta sobreajuste**. La restricción de profundidad máxima de los árboles y la técnica de ensamble de características aleatorias permitieron estructurar un modelo con una alta capacidad de generalización sobre los registros no observados.

2. **Capacidad Predictiva**:
   El F1-score ponderado y la Exactitud rondan los `0.66`. Este comportamiento es esperado en sistemas complejos de movilidad donde la variable objetivo representa una preferencia del pasajero (`payment_type`). Por lo que la consistencia entre Precisión y Recall refleja que el clasificador mantiene un equilibrio de errores (falsos positivos y falsos negativos) en todas las clases evaluadas.

#### 5.2. Modelo No Supervisado (K-means)

In [ ]:
# Tamaño de cada clúster
cluster_counts = M_clustered.groupBy("cluster").count().orderBy("cluster")
print("--- Tamaño de cada clúster ---")
cluster_counts.show()

# Medias de las variables de movilidad por clúster
cluster_profiles = M_clustered.groupBy("cluster").agg(
    F.mean("passenger_count").alias("avg_passengers"),
    F.mean("trip_distance").alias("avg_distance"),
    F.mean("fare_amount").alias("avg_fare"),
    F.mean("pickup_longitude").alias("avg_longitude"),
    F.mean("pickup_latitude").alias("avg_latitude"),
    F.count("cluster").alias("total_viajes")
).orderBy("cluster")

print("--- Perfil de Características Promedio por Clúster ---")
cluster_profiles.show()

### Interpretación de los Clústeres de Movilidad
A partir de los perfiles promedio de cada clúster, se pueden obtener las siguientes tipologías de viaje en la ciudad de Nueva York:
* **Clúster 0**: Representa la gran mayoría de los viajes urbanos estándar. Se caracteriza por distancias cortas (promedio de 1.5 - 2 millas) y tarifas bajas (menos de \$10 - \$12 dólares). Son traslados rápidos y locales dentro del centro metropolitano (Manhattan).
* **Clúster 1**: Agrupa los viajes de larga distancia y tarifas elevadas. Con una distancia promedio de 12 - 15 millas y una tarifa base promedio superior a \$40 dólares, este perfil se asocia con trayectos de enlace interurbano o traslados hacia aeropuertos.
* **Clúster 2**: Viajes de distancia media (entre 4 y 7 millas) y tarifas moderadas (alrededor de \$20 - \$25 dólares), que representan traslados entre distritos periféricos (e.g., Brooklyn o Queens a Manhattan).

El Coeficiente de Silueta obtenido para el $k$ óptimo valida que la división espacial y de tarifas entre estos tres grupos está muy definida, separando claramente los viajes cortos de uso diario, los viajes de distancia media y los viajes largos de tarifa fija.

### Reflexión
#### Fortalezas:
**Ausencia de Sesgo por Diseño**: El uso de muestreo estratificado proporcional mediante `sampleBy` preserva el comportamiento poblacional, eliminando sesgos de selección sistemáticos en el clasificador.
**Robustez**: Realizar la división train-test dentro de cada estrato garantiza que el modelo sea evaluado exactamente sobre la misma mezcla de patrones sobre la que fue entrenado.

#### Áreas de Oportunidad:
**Abordaje del Desbalance Extremo de Clases**: Dado que las clases de métodos de pago distintas de tarjeta de crédito y efectivo representan menos del 1% de los datos, se podría implementar un algoritmo de balanceo distribuido (como sobremuestreo SMOTE distribuido o submuestreo balanceado de clases mayoritarias) para mejorar el F1-score de clases minoritarias.

## 6. Bibliografía

* Chicco, D., & Jurman, G. (2020). The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation. *BMC Genomics*, 21(1), 1-13. https://doi.org/10.1186/s12864-019-6413-7
* Grandini, M., Bagli, E., & Visani, G. (2020). Metrics for multi-class classification: an overview. *arXiv preprint arXiv:2008.05756*. https://arxiv.org/abs/2008.05756
* Karau, H., & Warren, R. (2020). *High Performance Spark: Best Practices for Scaling and Optimizing Apache Spark*. O'Reilly Media.
* Lohr, S. L. (2021). *Sampling: Design and Analysis* (3rd ed.). CRC Press.
* Polak, A. (2023). *Scaling Machine Learning with Spark*. O'Reilly Media.
* Sarkar, D., & Nandi, A. (2022). *Practical Machine Learning with PySpark: Build and Deploy Machine Learning Pipelines*. Apress. https://doi.org/10.1007/978-1-4842-8191-8